# Notebook 05 — Smith-Waterman: Local Sequence Alignment

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 05 of 18  
**Time estimate:** 75 minutes

> **Pass-3 target:** Implement `smith_waterman()` from scratch without reference.
> The two changes from NW that make it local must be recallable from memory.

---
## Step 1 — Motivation

NW aligns entire sequences globally. But proteins often share only a conserved
domain — the rest of the sequence may be completely different. Local alignment finds
the best-scoring substring match: the most similar region within two sequences,
regardless of what's around it. Smith-Waterman (1981) solved this with two
modifications to NW.

---
## Step 2 — Intuition

In NW, the alignment must extend from corner to corner. If two sequences share only
a 20-amino-acid conserved domain, NW penalises the mismatched regions before and
after the domain, diluting the signal.

SW says: any cell can be the **start** of an alignment (score can't go below 0),
and the alignment **ends** at the highest-scoring cell, not the corner.

Two changes from NW:
1. Floor at 0: `F[i][j] = max(0, diagonal, up, left)`
2. Traceback starts at the **maximum** cell in the matrix, not the bottom-right corner.
   Stop when you hit a 0.

---
## Step 3 — Biological Background

Local alignment is the basis of BLAST. When you BLAST a protein query against a
database, BLAST finds the highest-scoring local alignment between your query and
each database sequence.

Applications:
- Finding conserved domains shared between otherwise-divergent proteins
- Aligning a short NGS read to a reference genome (though modern tools use
  specialized k-mer methods for speed)
- Detecting horizontal gene transfer (a bacterial gene locally similar to a eukaryotic one)

SW is guaranteed to find the **optimal** local alignment. BLAST trades optimality
for speed by using heuristics (k-mer seeding) — see NB08.

---
## Step 4 — Mathematical Explanation

**Recurrence (two changes from NW highlighted):**

$$H[i][j] = \max \begin{cases}
\mathbf{0} & \leftarrow \text{(SW change 1: floor at 0)} \\
H[i-1][j-1] + s(s_1[i], s_2[j]) & (\text{diagonal}) \\
H[i-1][j] + d & (\text{up}) \\
H[i][j-1] + d & (\text{left})
\end{cases}$$

**Initialization:** $H[0][j] = H[i][0] = 0$ for all $i, j$ (SW — no penalty for not starting at corners)

**Score:** $\max_{i,j} H[i][j]$ — the maximum value anywhere in the matrix.

**Traceback:** Start at $(i^*, j^*)$ where $H[i^*][j^*]$ is maximum. Follow back until you hit a cell with value 0. **(SW change 2)**

**Key consequence:** Every cell's value represents the optimal score of a local
alignment *ending* at position $(i,j)$. Setting the floor at 0 means the algorithm
always has the option to start fresh — a bad alignment up to that point doesn't
propagate.

In [ ]:
# Step 6 — Python Implementation
import numpy as np
from typing import Tuple


def smith_waterman(
    seq1: str,
    seq2: str,
    match: int = 2,
    mismatch: int = -1,
    gap: int = -1
) -> Tuple[int, str, str, int, int, int, int]:
    """
    Returns:
        score, aligned_seq1, aligned_seq2,
        start1, end1, start2, end2  (0-indexed positions in original sequences)
    """
    n, m = len(seq1), len(seq2)
    H = np.zeros((n + 1, m + 1), dtype=int)

    # No penalty for not starting at the corner (boundary stays 0)

    # Fill
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sc = match if seq1[i - 1] == seq2[j - 1] else mismatch
            H[i, j] = max(
                0,                          # floor at 0 — SW change 1
                H[i - 1, j - 1] + sc,      # diagonal
                H[i - 1, j] + gap,          # up
                H[i, j - 1] + gap           # left
            )

    # Find the maximum cell — SW change 2 (not necessarily bottom-right)
    max_score = int(H.max())
    max_pos = np.unravel_index(H.argmax(), H.shape)
    i, j = int(max_pos[0]), int(max_pos[1])

    # Traceback: stop at 0
    align1, align2 = [], []
    end1_idx = i - 1  # 0-indexed in original seq1
    end2_idx = j - 1  # 0-indexed in original seq2

    while i > 0 and j > 0 and H[i, j] > 0:  # stop at 0 — SW change 2
        sc = match if seq1[i - 1] == seq2[j - 1] else mismatch
        if H[i, j] == H[i - 1, j - 1] + sc:
            align1.append(seq1[i - 1])
            align2.append(seq2[j - 1])
            i -= 1; j -= 1
        elif H[i, j] == H[i - 1, j] + gap:
            align1.append(seq1[i - 1])
            align2.append('-')
            i -= 1
        else:
            align1.append('-')
            align2.append(seq2[j - 1])
            j -= 1

    start1_idx = i      # 0-indexed
    start2_idx = j

    a1 = ''.join(reversed(align1))
    a2 = ''.join(reversed(align2))
    return max_score, a1, a2, start1_idx, end1_idx, start2_idx, end2_idx


# Tests
print("=== Test 1: Fully identical (local = global) ===\n")
score, a1, a2, s1, e1, s2, e2 = smith_waterman('ATCG', 'ATCG')
print(f"Score: {score}")
print(f"seq1 aligned region: [{s1}:{e1+1}] = {a1}")
print(f"seq2 aligned region: [{s2}:{e2+1}] = {a2}")

print("\n=== Test 2: Conserved domain in divergent sequences ===\n")
seq1 = 'XXXXATCGATCGXXXX'  # conserved domain ATCGATCG in middle
seq2 = 'YYYYATCGATCGYYYY'
score, a1, a2, s1, e1, s2, e2 = smith_waterman(seq1, seq2)
print(f"Score: {score}")
print(f"seq1[{s1}:{e1+1}]: {a1}")
print(f"seq2[{s2}:{e2+1}]: {a2}")
print("SW correctly found only the conserved domain, ignoring the divergent flanks.")

print("\n=== Test 3: Classic SW example ===\n")
score, a1, a2, s1, e1, s2, e2 = smith_waterman('TGTTACGG', 'GGTTGACTA')
print(f"Score: {score}")
print(f"Aligned:")
print(f"  {a1}")
mid = ''.join('|' if a==b else (' ' if '-' in (a,b) else '.') for a,b in zip(a1,a2))
print(f"  {mid}")
print(f"  {a2}")

In [ ]:
# Validate against Biopython
from Bio import Align

aligner = Align.PairwiseAligner()
aligner.mode = 'local'
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -1
aligner.extend_gap_score = -1

test_pairs = [
    ('ATCG', 'ATCG'),
    ('TGTTACGG', 'GGTTGACTA'),
    ('AGTACGCA', 'TATGC'),
    ('ACGTACGT', 'TACG'),
]

print("Validation against Biopython local PairwiseAligner:")
print(f"{'Seq1':>15} {'Seq2':>15} {'Our score':>10} {'Bio score':>10} {'Match':>7}")
all_match = True
for s1, s2 in test_pairs:
    our_score, _, _, _, _, _, _ = smith_waterman(s1, s2)
    bio_score = aligner.score(s1, s2)
    ok = abs(our_score - bio_score) < 0.1
    if not ok:
        all_match = False
    print(f"{s1:>15} {s2:>15} {our_score:>10} {bio_score:>10.1f} {'OK' if ok else 'FAIL':>7}")

print(f"\nAll match: {all_match}")

In [ ]:
# Step 7 — Visualization: SW vs NW on the same pair
import matplotlib.pyplot as plt
import numpy as np


def nw_matrix(seq1, seq2, match=2, mismatch=-1, gap=-1):
    n, m = len(seq1), len(seq2)
    F = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): F[i,0] = i*gap
    for j in range(m+1): F[0,j] = j*gap
    for i in range(1,n+1):
        for j in range(1,m+1):
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            F[i,j] = max(F[i-1,j-1]+sc, F[i-1,j]+gap, F[i,j-1]+gap)
    return F


def sw_matrix(seq1, seq2, match=2, mismatch=-1, gap=-1):
    n, m = len(seq1), len(seq2)
    H = np.zeros((n+1, m+1), dtype=int)
    for i in range(1,n+1):
        for j in range(1,m+1):
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            H[i,j] = max(0, H[i-1,j-1]+sc, H[i-1,j]+gap, H[i,j-1]+gap)
    return H


# Use a sequence where local alignment makes a clear difference
seq1 = 'AAAAATCGATCGTTTT'
seq2 = 'ATCGATCG'

F = nw_matrix(seq1, seq2)
H = sw_matrix(seq1, seq2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, M, title, cmap in zip(
    axes, [F, H],
    [f'Needleman-Wunsch (global)\nScore at [n,m]: {F[len(seq1),len(seq2)]}',
     f'Smith-Waterman (local)\nMax score: {H.max()} (best local region)'],
    ['RdYlGn', 'Blues']
):
    n, m = len(seq1), len(seq2)
    ax.imshow(M, cmap=cmap, aspect='auto')
    ax.set_xticks(range(m+1))
    ax.set_xticklabels(['-']+list(seq2), fontsize=9)
    ax.set_yticks(range(n+1))
    ax.set_yticklabels(['-']+list(seq1), fontsize=9)
    for i in range(n+1):
        for j in range(m+1):
            ax.text(j, i, M[i,j], ha='center', va='center', fontsize=7)
    ax.set_title(title, fontsize=11)

# Mark max cell in SW matrix
max_pos = np.unravel_index(H.argmax(), H.shape)
axes[1].plot(max_pos[1], max_pos[0], 'r*', markersize=15, label='Max (start traceback)')
axes[1].legend(fontsize=8)

plt.suptitle(f'NW vs SW on "{seq1}" vs "{seq2}"\n'
             f'NW forces full alignment; SW finds the conserved core', fontsize=10)
plt.tight_layout()
plt.savefig('nw_vs_sw.png', dpi=150, bbox_inches='tight')
plt.show()

---
## The two changes: exact wording to memorise

| Aspect | Needleman-Wunsch (global) | Smith-Waterman (local) |
|--------|--------------------------|------------------------|
| Floor | No floor — cells can be negative | Floor at 0: `max(0, ...)` |
| Traceback start | Always $F[n][m]$ — bottom-right corner | Maximum cell in matrix |
| Traceback stop | Always $(0,0)$ | Stop when cell = 0 |
| Boundary init | $F[i][0] = i \cdot d$, $F[0][j] = j \cdot d$ | $H[i][0] = H[0][j] = 0$ |
| Use case | Full-length comparison | Finding conserved domains |

---
## Step 8 — Exercises

See `exercises/05_sw_exercises.md`.

1. Implement SW from scratch without looking at the code above.
2. On the pair ('HEAGAWGHEE', 'PAWHEAE'), run both NW and SW with match=5,
   mismatch=-3, gap=-4. Compare results — which is better for finding the
   conserved HE-containing region?
3. Write a function that returns all local alignments with score ≥ some threshold
   (not just the maximum). How does this relate to the idea of sub-optimal alignments?

---
## Step 10 — Quiz

1. State the two algorithmic differences between SW and NW.
2. If the SW matrix has all zeros, what does that mean biologically?
3. Is the SW local alignment always a substring of the NW global alignment? (Answer:
   no — explain why.)
4. Why does BLAST use local alignment rather than global?
5. What is the time complexity of SW?

---
## Step 12 — Reflection

> **Pass-3 rehearsal:** Implement SW from scratch, then compare to NW on one pair.
> Write the two change-points in a single sentence each before running any code.

> *[In one paragraph: explain to a biologist who has never heard of dynamic programming
> why SW finds the most similar region rather than the full sequence alignment.]*

---
## Step 13 — Papers Referenced

- Smith & Waterman (1981) "Identification of common molecular subsequences."
  *J Mol Biol* 147(1). The original paper. Pass-3 target.

---
*Next: `06_scoring_matrices_pam_blosum.ipynb`*